In [ ]:
import pandas as pd

# Insert a dataframe object
dfp = pd.DataFrame(pd.read_csv("Results_21Mar2022.csv"))
dfp.head()


In [ ]:
# Define variables

gender_group = ["female", "male"]
age_group = ["20-29", "30-39", "40-49", "50-59", "60-69", "70-79"]
diet_group = {
    "vegan": "Vegan",
    "veggie": "Veggie",
    "fish": "Fish",
    "meat50": "Meat <50",
    "meat": "Meat",
    "meat100": "Meat 100+",
}

raw_value_group = [
    "mean_ghgs",
    "mean_land",
    "mean_watscar",
    "mean_eut",
    "mean_ghgs_ch4",
    "mean_ghgs_n2o",
    "mean_bio",
    "mean_watuse",
    "mean_acid",
]

value_group = [
    ("ghgs", "GHG emissions (GreenHouse Gas) measured in kg"),
    (
        "ghgs_ch4",
        "GHG from CH<sub>4</sub> (Methane) emissions from livestock management measured in kg",
    ),
    (
        "ghgs_n2o",
        "GHG from N<sub>2</sub>O (Nitrous Oxide) emissions associated with fertilizer use",
    ),
    ("land", "Agricultural Land Use in square meters"),
    (
        "watuse",
        "Agricultural Water Usage in cubic meters (1 m<sup>3</sup> - 1,000 liters)",
    ),
    ("watscar", "Water Scarcity"),
    (
        "eut",
        "Eutrophication Potential– measured in g of PO<sub>4</sub>e, gPO<sub>4</sub>e",
    ),
    ("bio", "Biodiversity Impact–species extinction per day"),
    ("acid", "Acidification Potential"),
]

value_label = [
    ("ghgs", "GHG emissions"),
    ("ghgs_ch4", "GHG from CH<sub>4</sub>"),
    ("ghgs_n2o", "GHG from N<sub>2</sub>O"),
    ("land", "Land Usage"),
    ("watuse", "Water Usage"),
    ("watscar", "Water Scarcity"),
    ("eut", "Eutrophication Potential"),
    ("bio", "Biodiversity Impact"),
    ("acid", "Acidification Potential"),
]


data = dfp.copy()

# Generate new value group with its range 
# eg. "GHG emissions (GreenHouse Gas) measured in kg" -> "GHG emissions (GreenHouse Gas) measured in kg : Range 0 to 1"
value_group_w_range = value_group.copy()

for i, v in enumerate(value_group):
    vmin = data[[f"mean_{value_group[i][0]}"]].min().tolist()[0]
    vmax = data[[f"mean_{value_group[i][0]}"]].max().tolist()[0]

    value_group_w_range[i] = (value_group[i][0], f'{value_group[i][1]} : Range {vmin:.2f} to {vmax:.2f}')

# Overall Impact

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from sklearn.preprocessing import RobustScaler

def createMeanMatrix(df):
    # Create sum value of data of each environment measurement
    map = []
    for y, age  in enumerate(age_group):
        row = []
        for x, diet in enumerate(diet_group.keys()):
            f_data = df[(df["age_group"] == age) & (df["diet_group"] == diet)]
            sum_df = f_data[raw_value_group].sum()
            final_sum = 0
            for v in raw_value_group:
                value = sum_df[v]
                final_sum += value

            row.append(final_sum)
        map.append(row)

    return map


def plotSubGraphMean(sex, parent_figure, row, column):
    # Normalise data
    scaler = RobustScaler(quantile_range=(25, 75))
    norm_data = data.copy()
    norm_data[raw_value_group] = scaler.fit_transform(data[raw_value_group])

    norm_data = norm_data[(norm_data["sex"] == sex)]

    # Create heatmap data matrix
    z_map = createMeanMatrix(norm_data)

    # Create heatmap
    parent_figure.add_trace(
        go.Heatmap(
            z=z_map,
            hoverinfo="none",
            coloraxis="coloraxis",
        ),
        row=row,
        col=column,
    )

    # Create Borders
    # Vertical lines
    for x in range(0, len(diet_group) + 1):
        parent_figure.add_shape(
            type="line",
            x0=x - 0.5,
            x1=x - 0.5,  # Vertical position
            y0=-0.5,
            y1=len(age_group) - 0.5,  # Cover all rows
            line=dict(color="white", width=1),
            row=row,
            col=column,
        )

    # Horizontal lines
    for x in range(0, len(age_group) + 1):
        parent_figure.add_shape(
            dict(
                type="line",
                x0=-0.5,
                x1=len(diet_group) - 0.5,  # Cover all columns
                y0=x - 0.5,
                y1=x - 0.5,  # Horizontal position
                line=dict(color="white", width=1),
            ),
            row=row,
            col=column,
        )

    return max(max(z_map))


def plotGraphMean():
    # key, title = value
    # Create a parent figure with 1x2 subplots (for female and male)
    parent_fig = make_subplots(rows=1, cols=2, subplot_titles=["Female", "Male"])
    max_f = plotSubGraphMean("female", parent_fig, row=1, column=1)
    max_m = plotSubGraphMean("male", parent_fig, row=1, column=2)
    max_z = max(max_f, max_m)

    # Setup parent figure layout
    parent_fig.update_layout(
        title="Enviromental Impact (Relative)",
        title_x=0.5,
        showlegend=False,
        coloraxis=dict(
            colorscale="thermal",
            showscale=True,
            colorbar=dict(
                title="Impact (%)",  # Colorbar label
                tickvals=np.linspace(0, max_z, 3),
                ticktext=["0%", "50%", "100%"],  # Custom labels
            ),
        ),
        height=400,
        width=750,
    )

    parent_fig.update_xaxes(
        tickvals=np.arange(len(diet_group)),
        ticktext=[diet for diet in diet_group.values()],
    )
    parent_fig.update_yaxes(
        tickvals=np.arange(len(age_group)),
        ticktext=[age.capitalize() for age in age_group],
    )

    parent_fig.show()

In [ ]:
plotGraphMean()

# All Measurements (Multivariate)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from sklearn.preprocessing import MinMaxScaler


def plotSubGraph(sex, selected_values, parent_figure, row, column):
    # Normalise data
    scaler = MinMaxScaler()
    norm_data = data.copy()
    norm_data[raw_value_group] = scaler.fit_transform(data[raw_value_group])

    # Loop through the Age Groups and Diet Groups to create boxes
    for y, age in enumerate(age_group):  # Use age group as y axis
        for x, diet in enumerate(diet_group.keys()):  # Use diet group as x axis
            # Filtering data
            group_df = data[
                (data["sex"] == sex)
                & (data["age_group"] == age)
                & (data["diet_group"] == diet)
            ]
            group_data = group_df.values

            norm_group_df = norm_data[
                (norm_data["sex"] == sex)
                & (norm_data["age_group"] == age)
                & (norm_data["diet_group"] == diet)
            ]

            # Create Heatmap with data of each run and value for every environmental impact
            startPoint = x - 0.5
            unitWidth = 1 / len(group_data)
            unitHeight = 1 / len(selected_values)

            for i, group in enumerate(reversed(selected_values)):
                value_index = [v[0] for v in value_group].index(group)
                value = value_group[value_index]
                key = value[0]  # Environmental impact key
                label = value_label[value_index][1]  # Environmental impact label

                # Define text when hovering on sample (data point)
                text_values = []

                for j in range(len(group_df.values)):  # Columns (diet groups)
                    sample = group_df.iloc[j]
                    text = f"<br><b>{label}</b><br>Run ID: {sample['mc_run_id']}<br>Age: {age}<br>Diet: {diet}<br>Mean: {sample[f'mean_{key}']}<br>SD: {sample[f'sd_{key}']}"
                    text_values.append(text)

                startPointY = y - 0.5 + (i * unitHeight)
                parent_figure.add_trace(
                    go.Heatmap(
                        x=[
                            startPoint,  # first point position X
                            startPoint + unitWidth,  # second point position X
                        ],
                        y=[
                            startPointY,  # first point position Y
                            startPointY + unitHeight,  # second point position Y
                        ],
                        z=[norm_group_df[f"mean_{key}"].values],
                        text=[text_values],
                        hoverinfo="text",
                        coloraxis="coloraxis",
                    ),
                    row=row,
                    col=column,
                )

    # Create Borders
    # Vertical lines
    for y in range(0, len(diet_group) + 1):
        parent_figure.add_shape(
            type="line",
            x0=y - 0.5 - (unitWidth / 2),
            x1=y - 0.5 - (unitWidth / 2),  # Vertical position
            y0=-0.5,
            y1=len(age_group) - 0.5,  # Cover all rows
            line=dict(color="white", width=1),
            row=row,
            col=column,
        )

    # Horizontal lines
    for y in range(0, len(age_group) + 1):
        parent_figure.add_shape(
            dict(
                type="line",
                x0=-0.5 - (unitWidth / 2),
                x1=len(diet_group) - 0.5 - (unitWidth / 2),  # Cover all columns
                y0=y - 0.5,
                y1=y - 0.5,  # Horizontal position
                line=dict(color="white", width=1),
            ),
            row=row,
            col=column,
        )


def plotGraph(selected_values):
    # key, title = value
    # Create a parent figure with 1x2 subplots (for female and male)
    parent_fig = make_subplots(rows=1, cols=2, subplot_titles=["Female", "Male"])
    plotSubGraph("female", selected_values, parent_fig, row=1, column=1)
    plotSubGraph("male", selected_values, parent_fig, row=1, column=2)

    # Setup parent figure layout
    parent_fig.update_layout(
        title="Enviromental Impact (Detail)",
        title_x=0.5,
        showlegend=False,
        coloraxis=dict(
            colorscale="thermal",
            showscale=True,
            colorbar=dict(title="Normalised Scale"),
        ),
        height=500,
        width=1400,
    )

    parent_fig.update_xaxes(
        tickvals=np.arange(len(diet_group)),
        ticktext=[diet for diet in diet_group.values()],
    )
    parent_fig.update_yaxes(
        tickvals=np.arange(len(age_group)),
        ticktext=[age.capitalize() for age in age_group],
    )

    return parent_fig

In [ ]:
from dash import html


def convertHtml(text):
    # Replace <sub></sub> tag in text with Html element
    text = text.replace("/sub", "sub")
    text_split = text.split("<sub>")

    elements = []
    for i, text in enumerate(text_split):
        if i % 2 == 1:
            element = [html.Sub([text])]
        else:
            element = convertSupHtml(text)

        elements += element

    return html.Span(elements)


def convertSupHtml(text):
    # Replace <sup></sup> tag in text with Html element
    text = text.replace("/sup", "sup")
    text_split = text.split("<sup>")

    elements = []
    for i, text in enumerate(text_split):
        if i % 2 == 1:
            element = html.Sup([text])
        else:
            element = text

        elements.append(element)

    return elements

In [ ]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output

# Dash App
app = dash.Dash(__name__)

app.layout = html.Div(
    [
        html.Br(),
        dcc.Checklist(
            id="value-group-selector",
            options=[
                {"label": convertHtml(name), "value": key}
                for key, name in value_group_w_range
            ],
            value=['ghgs'],  # Default selection

        ),
        html.Br(),
        dcc.Graph(id="interactive-heatmap"),
    ]
)


@app.callback(
    Output("interactive-heatmap", "figure"),
    Input("value-group-selector", "value"),
)
def update_graph(selected_values):
    return plotGraph(selected_values)


if __name__ == "__main__":
    app.run(debug=True)
    # app.run(debug=True, jupyter_mode="external") # Create link to view on browser

# Single measurement visualisation for each environmental impact

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np


def plotSubGraphOne(sex, key, parent_figure, row, column):
    # Loop through the Age Groups and Diet Groups to create boxes
    for y, age in enumerate(age_group):  # Use age group as y axis
        for x, diet in enumerate(diet_group.keys()):  # Use diet group as x axis
            # Filtering data
            group_df = data[
                (data["sex"] == sex)
                & (data["age_group"] == age)
                & (data["diet_group"] == diet)
            ]
            group_data = group_df.values

            # Define text when hovering on sample (data point)
            text_values = []
            for i in range(len(group_df.values)):  # Columns (diet groups)
                sample = group_df.iloc[i]
                text = f"<br>Run ID: {sample['mc_run_id']}<br>Age: {age}<br>Diet: {diet}<br>Mean: {sample[f'mean_{key}']}<br>SD: {sample[f'sd_{key}']}"
                text_values.append(text)

            # Create Heatmap with data of each run and value
            startPoint = x - 0.5
            size = len(group_data)
            unitWidth = 1 / size
            parent_figure.add_trace(
                go.Heatmap(
                    x=[
                        startPoint,  # first point position
                        startPoint + unitWidth,  # second point position
                    ],
                    y=[y],
                    z=[group_df[f'mean_{key}'].values],
                    text=[text_values],
                    hoverinfo="text",
                    coloraxis="coloraxis",
                ),
                row=row,
                col=column,
            )

    # Create Borders
    # Vertical lines
    for y in range(0, len(diet_group) + 1):
        parent_figure.add_shape(
            type="line",
            x0=y - 0.5 - (unitWidth / 2),
            x1=y - 0.5 - (unitWidth / 2),  # Vertical position
            y0=-0.5,
            y1=len(age_group) - 0.5,  # Cover all rows
            line=dict(color="white", width=1),
            row=row,
            col=column,
        )

    # Horizontal lines
    for y in range(0, len(age_group) + 1):
        parent_figure.add_shape(
            dict(
                type="line",
                x0=-0.5 - (unitWidth / 2),
                x1=len(diet_group) - 0.5 - (unitWidth / 2),  # Cover all columns
                y0=y - 0.5,
                y1=y - 0.5,  # Horizontal position
                line=dict(color="white", width=1),
            ),
            row=row,
            col=column,
        )


def plotGraphOne(value):
    key, title = value
    # Create a parent figure with 1x2 subplots (for female and male)
    parent_fig = make_subplots(rows=1, cols=2, subplot_titles=["Female", "Male"])
    plotSubGraphOne("female", key, parent_fig, row=1, column=1)
    plotSubGraphOne("male", key, parent_fig, row=1, column=2)

    # Setup parent figure layout
    parent_fig.update_layout(
        title=title,
        title_x=0.5,
        showlegend=False,
        coloraxis=dict(
            colorscale="thermal",
            showscale=True,
        ),
        height=400,
        width=750,
    )

    parent_fig.update_xaxes(tickvals=np.arange(len(diet_group)), ticktext=[diet for diet in diet_group.values()])
    parent_fig.update_yaxes(tickvals=np.arange(len(age_group)), ticktext=[age.capitalize() for age in age_group])

    parent_fig.show()


In [ ]:
for value in value_group:
    plotGraphOne(value)
